## 1. Environment Setup

In [1]:
from google.colab import drive, userdata
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import sys

drive.mount('/content/drive')

DATA_ROOT = Path("/content/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset")
PED1_PATH = DATA_ROOT / "UCSDped1"
PED2_PATH = DATA_ROOT / "UCSDped2"
ZIP_PATH = Path("/content/drive/MyDrive/SurveillanceAnomalyDetection/UCSD_Anomaly_Dataset.zip")
EXTRACT_PATH = Path("/content/UCSD_Anomaly_Dataset")

if not (PED1_PATH.exists() and PED2_PATH.exists()):
  with ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_PATH)

GH_TOKEN = userdata.get("GH_PAT")
REPO_DIR = Path("/content/drive/MyDrive/SurveillanceAnomalyDetection/repo")
REPO_URL = f"https://{GH_TOKEN}@github.com/Rishabh-G-Shetye/SurveillanceAnomalyDetection.git"
if not REPO_DIR.exists():
  !git clone {REPO_URL} "{REPO_DIR}"
%cd "{REPO_DIR}"

sys.path.insert(0, str(REPO_DIR / "src"))

df = pd.read_csv(REPO_DIR / "data" / "metadata.csv")
from data.ucsd_dataset import PreprocessConfig, UCSDClipDataset

ped1_config = PreprocessConfig(target_size=(128, 128), window_length=8, stride=4)
train_ds = UCSDClipDataset(df, DATA_ROOT, "Ped1", "Train", ped1_config)
test_ds = UCSDClipDataset(df, DATA_ROOT, "Ped1", "Test", ped1_config)
print("Train clips:", len(train_ds), "| Test clips:", len(test_ds))

Mounted at /content/drive
/content/drive/MyDrive/SurveillanceAnomalyDetection/repo
Train clips: 1666 | Test clips: 1762


## 2. Module Directories  

In [2]:
import os
for sub in ["models", "training", "evaluation", "utils"]:
  os.makedirs(REPO_DIR / "src" / sub, exist_ok=True)
  (REPO_DIR / "src" / sub / "__init__.py").touch()

## 2b. Persistency

In [3]:
%%writefile "{REPO_DIR}/src/utils/persistence.py"
"""
Model/result persistence -- local version (replaces Colab Drive paths).

Checkpoints are saved under PROJECT_ROOT/models/
Results and figures under PROJECT_ROOT/outputs/
"""
import json
from pathlib import Path
import numpy as np
import torch

# Project root is two levels up from this file (src/utils/persistence.py -> project root)
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent

CKPT_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "logs"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"


def save_checkpoint(model: torch.nn.Module, name: str) -> Path:
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    path = CKPT_DIR / f"{name}.pt"
    torch.save(model.state_dict(), path)
    print(f"  Checkpoint saved: {path}")
    return path


def load_checkpoint(model: torch.nn.Module, name: str, device=None) -> torch.nn.Module:
    path = CKPT_DIR / f"{name}.pt"
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return model.to(device)


def save_results(name: str, results: dict, scores: np.ndarray, labels: np.ndarray) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_DIR / f"{name}_metrics.json", "w") as f:
        json.dump(results, f, indent=2, default=float)
    np.savez(RESULTS_DIR / f"{name}_scores.npz", scores=scores, labels=labels)
    print(f"  Results saved: {RESULTS_DIR / name}")


def load_results(name: str):
    with open(RESULTS_DIR / f"{name}_metrics.json") as f:
        results = json.load(f)
    npz = np.load(RESULTS_DIR / f"{name}_scores.npz")
    return results, npz["scores"], npz["labels"]


def try_load_results(name: str):
    """Like load_results, but returns None if the artifact doesn't exist."""
    try:
        return load_results(name)
    except FileNotFoundError:
        return None


def save_metrics_only(name: str, results: dict) -> Path:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    path = RESULTS_DIR / f"{name}_metrics.json"
    with open(path, "w") as f:
        json.dump(results, f, indent=2, default=float)
    return path


def save_figure(fig, name: str) -> Path:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURES_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Figure saved: {path}")
    return path


Overwriting /content/drive/MyDrive/SurveillanceAnomalyDetection/repo/src/utils/persistence.py


## 3. Base Model Interface (`BaseAnomalyModel`)

### Design Contract:
All our models inherit from `BaseAnomalyModel` so the same trainer and evaluation functions work across all architectures.
- `forward(clip)`: Generates reconstructions or predictions.
- `compute_loss(clip)`: Unsupervised training loss on normal video clips.
- `per_frame_anomaly_score(clip)`: Returns per-frame errors (`(B, T)` for autoencoders, `(B,)` for frame prediction) so evaluation can evaluate frame-by-frame.


In [4]:
%%writefile "{REPO_DIR}/src/models/base.py"
"""
Base interface for all anomaly detection models in this project.

Shared contract so our training loop, evaluators, and visualizers work with
any model (ConvAE, ConvLSTM, Transformer, FramePrediction, MemAE) without rewriting code.
"""

from abc import ABC, abstractmethod
import torch
import torch.nn as nn


class BaseAnomalyModel(nn.Module, ABC):
    """Abstract base class that all 5 anomaly detection models inherit from."""

    @abstractmethod
    def forward(self, clip: torch.Tensor) -> torch.Tensor:
        # clip: (B, T, C, H, W)
        # returns reconstructed clip (B, T, C, H, W) or predicted frame (B, C, H, W)
        raise NotImplementedError

    @abstractmethod
    def compute_loss(self, clip: torch.Tensor) -> torch.Tensor:
        # Unsupervised loss on normal training video (e.g. MSE, entropy penalty)
        raise NotImplementedError

    @abstractmethod
    def per_frame_anomaly_score(self, clip: torch.Tensor) -> torch.Tensor:
        # Returns frame-level reconstruction/prediction errors.
        # (B, T) for autoencoders, (B,) for frame prediction.
        # Higher score = higher error = more likely an anomaly.
        raise NotImplementedError

    def anomaly_score(self, clip: torch.Tensor) -> torch.Tensor:
        # Summarize frame scores into a single clip score
        scores = self.per_frame_anomaly_score(clip)
        agg = getattr(self, "score_agg", "mean")
        if scores.ndim == 2:
            return scores.max(dim=1).values if agg == "max" else scores.mean(dim=1)
        return scores


Overwriting /content/drive/MyDrive/SurveillanceAnomalyDetection/repo/src/models/base.py


## 4. Model 1 - Baseline Spatial ConvAE

### Why No Skip Connections?
- We tested ConvAE with and without U-Net skip connections.
- When skip connections are enabled, low-level edge features (like bicycle wheels or carts) bypass the latent bottleneck and copy straight to the decoder. The model reconstructs anomalies too easily, giving poor anomaly detection AUC.
- Setting `use_skip=False` forces an information bottleneck so unseen anomalous patterns incur high reconstruction error.


In [5]:
%%writefile "{REPO_DIR}/src/models/conv_ae.py"
"""
Baseline Convolutional Autoencoder (ConvAE) for Video Anomaly Detection.

Compresses each frame down to a 16x16 spatial bottleneck and tries to reconstruct it.
Normal pedestrians are reconstructed cleanly, while unseen anomalous objects
(bicycles, skaters, carts) have high reconstruction error.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

from .base import BaseAnomalyModel


class ConvAE(BaseAnomalyModel):
    def __init__(
        self,
        in_channels: int = 1,
        latent_channels: int = 64,
        score_agg: str = "mean",
        use_skip: bool = False,
    ):
        super().__init__()
        assert score_agg in ("mean", "max")
        self.score_agg = score_agg
        # Note: keep use_skip=False by default! Skip connections pass low-level edges
        # straight to the decoder, letting anomalous bikes/carts reconstruct too well.
        self.use_skip = use_skip

        # Encoder: 128x128 -> 64x64 -> 32x32 -> 16x16
        self.enc1 = nn.Sequential(nn.Conv2d(in_channels, 32, 4, 2, 1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(64, latent_channels, 4, 2, 1), nn.ReLU())

        # Decoder: 16x16 -> 32x32 -> 64x64 -> 128x128
        dec2_in = 64 * 2 if use_skip else 64
        dec1_in = 32 * 2 if use_skip else 32
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(latent_channels, 64, 4, 2, 1), nn.ReLU())
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(dec2_in, 32, 4, 2, 1), nn.ReLU())
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(dec1_in, in_channels, 4, 2, 1), nn.Tanh())

    def _forward_frame(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        d3 = self.dec3(e3)
        d2 = self.dec2(torch.cat([d3, e2], dim=1) if self.use_skip else d3)
        d1 = self.dec1(torch.cat([d2, e1], dim=1) if self.use_skip else d2)
        return d1

    def forward(self, clip: torch.Tensor) -> torch.Tensor:
        # Reshape (B, T, C, H, W) to (B*T, C, H, W) to pass through 2D CNN layers
        B, T, C, H, W = clip.shape
        recon = self._forward_frame(clip.view(B * T, C, H, W))
        return recon.view(B, T, C, H, W)

    def compute_loss(self, clip: torch.Tensor) -> torch.Tensor:
        # Mean Squared Error between input and reconstruction
        recon = self.forward(clip)
        return F.mse_loss(recon, clip)

    def per_frame_anomaly_score(self, clip: torch.Tensor) -> torch.Tensor:
        # Compute MSE per individual frame in the clip -> returns (B, T)
        with torch.no_grad():
            recon = self.forward(clip)
            return ((recon - clip) ** 2).mean(dim=(2, 3, 4))


Overwriting /content/drive/MyDrive/SurveillanceAnomalyDetection/repo/src/models/conv_ae.py


In [6]:
from models.conv_ae import ConvAE
model = ConvAE()

## 5. Training Loop & Regularization

### Training Strategy:
- Optimizer: Adam with `weight_decay=1e-5` to regularize against static surveillance backgrounds.
- Learning Rate Scheduler: `ReduceLROnPlateau(factor=0.5, patience=2)` cuts LR when training loss plateaus.


In [7]:
%%writefile "{REPO_DIR}/src/training/trainer.py"
"""
Generic training loop shared by every BaseAnomalyModel implementation.
"""
from typing import Optional
import torch
from torch.utils.data import DataLoader


def train_model(
    model: torch.nn.Module,
    train_loader: DataLoader,
    num_epochs: int = 30,
    lr: float = 1e-3,
    device: Optional[str] = None,
    patience: int = 5,
) -> list:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )

    history = []
    best_loss, epochs_without_improvement = float("inf"), 0

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            clip = batch["clip"].to(device)
            optimizer.zero_grad()
            loss = model.compute_loss(clip)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * clip.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        history.append(epoch_loss)
        scheduler.step(epoch_loss)
        print(
            f"Epoch {epoch+1}/{num_epochs} - loss: {epoch_loss:.6f}"
            f" - lr: {optimizer.param_groups[0]['lr']:.2e}"
        )

        if epoch_loss < best_loss - 1e-6:
            best_loss, epochs_without_improvement = epoch_loss, 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping: no improvement for {patience} epochs.")
                break
    return history


Overwriting /content/drive/MyDrive/SurveillanceAnomalyDetection/repo/src/training/trainer.py


## 6. Evaluation Metrics (Literature Benchmark Protocol)

### Matching Published UCSD Papers (Hasan et al., Liu et al.):
1. **Per-Sequence Normalization:** We normalize scores to `[0, 1]` per test video to cancel out lighting/exposure differences between camera takes.
2. **Temporal Smoothing:** Mild 1D Gaussian filter removes single-frame sensor jitter.
3. **Optimal Thresholding:** We find the threshold on the ROC curve that maximizes F1 score.
4. **Event-Level IoU Localization:** Measures temporal overlap (IoU >= 0.1) to test if full anomaly events are detected.


In [8]:
%%writefile "{REPO_DIR}/src/evaluation/metrics.py"
"""
Evaluation metrics for Video Anomaly Detection.

Implements:
- AUC-ROC (Area Under the Receiver Operating Characteristic Curve)
- EER (Equal Error Rate)
- Optimal thresholding (F1-maximizing / Youden's J statistic)
- Precision, Recall, F1-Score
- Frame-level evaluation across all test videos
- Event-level evaluation (temporal localization via 1D IoU segment overlap)
- Per-sequence min-max score normalization and temporal smoothing (standard UCSD benchmark protocol)
- Computational efficiency benchmarking (ms/frame and FPS)
"""

import time
from collections import defaultdict

import numpy as np
import torch
from torch.utils.data import DataLoader
from scipy.ndimage import gaussian_filter1d
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support


def compute_eer(labels: np.ndarray, scores: np.ndarray) -> float:
    """Equal Error Rate: threshold where false positive rate equals false negative rate."""
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[eer_idx] + fnr[eer_idx]) / 2)


def evaluate_scores(labels: np.ndarray, scores: np.ndarray, threshold: float = None) -> dict:
    """
    Compute comprehensive metrics given binary ground-truth labels and anomaly scores.
    If threshold is None, finds the optimal threshold that maximizes F1 score.
    """
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    # Handle edge case of single-class labels in small tests
    if len(np.unique(labels)) < 2:
        return {
            "auc_roc": 1.0 if (labels == (scores >= 0.5)).all() else 0.5,
            "eer": 0.0,
            "precision": 1.0,
            "recall": 1.0,
            "f1": 1.0,
            "threshold_used": 0.5,
        }

    auc = roc_auc_score(labels, scores)
    eer = compute_eer(labels, scores)

    fpr, tpr, thresholds = roc_curve(labels, scores)

    # Find optimal threshold by maximizing F1 on candidate thresholds
    if threshold is None:
        # Sample candidate thresholds from percentiles
        candidate_threshs = np.unique(
            np.percentile(scores, np.linspace(5, 95, 50))
        )
        best_f1 = -1.0
        best_thresh = float(np.median(scores))

        for t in candidate_threshs:
            preds = (scores >= t).astype(int)
            p, r, f1, _ = precision_recall_fscore_support(
                labels, preds, average="binary", zero_division=0
            )
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = float(t)
        threshold = best_thresh

    preds = (scores >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    return {
        "auc_roc": float(auc),
        "eer": float(eer),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "threshold_used": float(threshold),
    }


def _to_segments(binary_arr: np.ndarray) -> list[tuple[int, int]]:
    """Contiguous runs of True in a 1D boolean array -> [(start, end), ...]."""
    segments, start = [], None
    for i, v in enumerate(binary_arr):
        if v and start is None:
            start = i
        elif not v and start is not None:
            segments.append((start, i))
            start = None
    if start is not None:
        segments.append((start, len(binary_arr)))
    return segments


def _iou(a: tuple[int, int], b: tuple[int, int]) -> float:
    """1D temporal Intersection over Union between two intervals."""
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union else 0.0


def event_level_accuracy(
    frame_labels: np.ndarray,
    frame_scores: np.ndarray,
    threshold: float,
    iou_thresh: float = 0.1,
) -> dict:
    """
    Event-level evaluation: groups contiguous ground-truth anomalous frames into
    events, and contiguous above-threshold predictions into predicted events.
    An event is detected if 1D IoU >= iou_thresh.
    """
    gt_segments = _to_segments(np.asarray(frame_labels).astype(bool))
    pred_segments = _to_segments(np.asarray(frame_scores) >= threshold)

    matched_gt = set()
    tp = 0
    for p in pred_segments:
        for i, g in enumerate(gt_segments):
            if i not in matched_gt and _iou(p, g) >= iou_thresh:
                tp += 1
                matched_gt.add(i)
                break

    precision = tp / len(pred_segments) if pred_segments else 0.0
    recall = tp / len(gt_segments) if gt_segments else (1.0 if not pred_segments else 0.0)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {
        "event_precision": float(precision),
        "event_recall": float(recall),
        "event_f1": float(f1),
        "num_gt_events": len(gt_segments),
        "num_pred_events": len(pred_segments),
    }


def smooth_scores(scores: np.ndarray, sigma: float = 1.5) -> np.ndarray:
    """Applies 1D Gaussian filter along temporal dimension to suppress high-frequency frame noise."""
    if len(scores) < 3 or sigma <= 0:
        return scores
    return gaussian_filter1d(scores, sigma=sigma, mode="nearest")


def frame_and_event_level_eval(
    model: torch.nn.Module,
    dataset,
    device: str | None = None,
    batch_size: int = 32,
    normalize_per_seq: bool = True,
    smooth: bool = True,
    iou_thresh: float = 0.1,
) -> dict:
    """
    High-performance batched frame-level and event-level evaluation.
    
    1. Runs model.per_frame_anomaly_score in batched DataLoader.
    2. Maps scores back to individual frames, averaging overlapping windows.
    3. Normalizes scores per video sequence (standard UCSD benchmark protocol).
    4. Computes pooled frame-level AUC-ROC, EER, F1, and sequence-averaged event metrics.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    seq_frame_scores = defaultdict(lambda: defaultdict(list))
    seq_frame_labels = defaultdict(dict)

    is_prediction_model = (model.__class__.__name__ == "FramePredictionNet")

    with torch.no_grad():
        for batch in loader:
            clips = batch["clip"].to(device)
            # frame_scores shape: (B, T) for reconstruction, (B,) for prediction
            scores = model.per_frame_anomaly_score(clips).cpu().numpy()
            frame_indices = batch["frame_indices"].numpy()  # (B, T)
            labels = batch["frame_labels"].numpy()          # (B, T)
            seq_names = batch["seq_name"]                   # list of strings, length B

            B = len(seq_names)
            for b in range(B):
                s_name = seq_names[b]
                if is_prediction_model:
                    # Prediction score belongs strictly to the predicted frame (last in window)
                    f_idx = int(frame_indices[b, -1])
                    seq_frame_scores[s_name][f_idx].append(float(scores[b]))
                    seq_frame_labels[s_name][f_idx] = int(labels[b, -1])
                else:
                    # Reconstruction errors map frame-by-frame
                    T = frame_indices.shape[1]
                    for t in range(T):
                        f_idx = int(frame_indices[b, t])
                        seq_frame_scores[s_name][f_idx].append(float(scores[b, t]))
                        seq_frame_labels[s_name][f_idx] = int(labels[b, t])

    # Aggregate per sequence
    all_scores, all_labels = [], []
    event_precisions, event_recalls, event_f1s = [], [], []

    for seq_name in sorted(seq_frame_scores.keys()):
        frame_map = seq_frame_scores[seq_name]
        f_idxs = sorted(frame_map.keys())

        # Average overlapping window scores for each frame
        raw_s = np.array([np.mean(frame_map[idx]) for idx in f_idxs], dtype=float)
        f_labels = np.array([seq_frame_labels[seq_name][idx] for idx in f_idxs], dtype=int)

        # Standard benchmark per-sequence min-max scaling
        if normalize_per_seq:
            s_min, s_max = raw_s.min(), raw_s.max()
            norm_s = (raw_s - s_min) / (s_max - s_min + 1e-8)
        else:
            norm_s = raw_s

        # Temporal smoothing
        if smooth:
            final_s = smooth_scores(norm_s, sigma=1.5)
        else:
            final_s = norm_s

        all_scores.append(final_s)
        all_labels.append(f_labels)

        # Sequence-level event accuracy
        seq_threshold = float(np.median(final_s))
        ev = event_level_accuracy(f_labels, final_s, seq_threshold, iou_thresh=iou_thresh)
        event_precisions.append(ev["event_precision"])
        event_recalls.append(ev["event_recall"])
        event_f1s.append(ev["event_f1"])

    pooled_scores = np.concatenate(all_scores)
    pooled_labels = np.concatenate(all_labels)

    frame_results = evaluate_scores(pooled_labels, pooled_scores)

    return {
        **{f"frame_{k}": v for k, v in frame_results.items()},
        "event_precision_avg": float(np.mean(event_precisions)),
        "event_recall_avg": float(np.mean(event_recalls)),
        "event_f1_avg": float(np.mean(event_f1s)),
        "total_evaluated_frames": len(pooled_labels),
        "total_anomalous_frames": int(pooled_labels.sum()),
        "total_evaluated_sequences": len(seq_frame_scores),
        "pooled_scores": pooled_scores,
        "pooled_labels": pooled_labels,
    }


def measure_inference_time(
    model: torch.nn.Module,
    sample_clip: torch.Tensor,
    device: str | None = None,
    n_runs: int = 50,
) -> dict:
    """
    Benchmark inference latency per clip and per frame, and compute FPS.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    clip = sample_clip.unsqueeze(0).to(device)

    with torch.no_grad():
        # Warmup
        for _ in range(5):
            model.per_frame_anomaly_score(clip)
        if device == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        for _ in range(n_runs):
            model.per_frame_anomaly_score(clip)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

    ms_per_clip = (elapsed / n_runs) * 1000
    T = clip.shape[1]
    ms_per_frame = ms_per_clip / T
    fps = 1000.0 / ms_per_frame if ms_per_frame > 0 else 0.0

    return {
        "ms_per_clip": float(ms_per_clip),
        "ms_per_frame": float(ms_per_frame),
        "fps": float(fps),
    }


Overwriting /content/drive/MyDrive/SurveillanceAnomalyDetection/repo/src/evaluation/metrics.py


## 7. Train ConvAE and Evaluate

In [9]:
import itertools
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from models.conv_ae import ConvAE
from training.trainer import train_model
from evaluation.metrics import evaluate_scores
from utils.persistence import save_checkpoint, save_results

device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

ablation_results = {}
ablation_models = {}
for use_skip, score_agg in itertools.product([False, True], ["mean", "max"]):
    name = f"skip={use_skip}, agg={score_agg}"
    m = ConvAE(use_skip=use_skip, score_agg=score_agg)
    train_model(m, train_loader, num_epochs=15, patience=3)
    m.eval()
    scores, labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            clip = batch["clip"].to(device)
            scores.append(m.anomaly_score(clip).cpu().numpy())
            labels.append(batch["clip_label"].numpy())
    scores, labels = np.concatenate(scores), np.concatenate(labels)
    ablation_results[name] = evaluate_scores(labels, scores)
    ablation_models[name] = m

    # Persist every ablation run individually, so a runtime reset never
    # forces a full re-run of the whole ablation just to get one config back.
    safe_name = name.replace(" ", "").replace(",", "_")
    save_checkpoint(m, f"conv_ae_{safe_name}")
    save_results(f"conv_ae_{safe_name}", ablation_results[name], scores, labels)

    print(name, "->", ablation_results[name])

pd.DataFrame(ablation_results).T[["auc_roc", "eer", "f1"]]

Epoch 1/15 - loss: 0.055272 - lr: 1.00e-03
Epoch 2/15 - loss: 0.014597 - lr: 1.00e-03
Epoch 3/15 - loss: 0.009155 - lr: 1.00e-03
Epoch 4/15 - loss: 0.006937 - lr: 1.00e-03
Epoch 5/15 - loss: 0.005644 - lr: 1.00e-03
Epoch 6/15 - loss: 0.004826 - lr: 1.00e-03
Epoch 7/15 - loss: 0.004177 - lr: 1.00e-03
Epoch 8/15 - loss: 0.003697 - lr: 1.00e-03
Epoch 9/15 - loss: 0.003348 - lr: 1.00e-03
Epoch 10/15 - loss: 0.003114 - lr: 1.00e-03
Epoch 11/15 - loss: 0.002943 - lr: 1.00e-03
Epoch 12/15 - loss: 0.002722 - lr: 1.00e-03
Epoch 13/15 - loss: 0.002599 - lr: 1.00e-03
Epoch 14/15 - loss: 0.002477 - lr: 1.00e-03
Epoch 15/15 - loss: 0.002364 - lr: 1.00e-03
skip=False, agg=mean -> {'auc_roc': np.float64(0.6319944828898243), 'eer': 0.4298968136094483, 'precision': 0.2054483541430193, 'recall': 0.5638629283489096, 'f1': 0.3011647254575707, 'threshold_used': 0.004548028111457825}
Epoch 1/15 - loss: 0.076906 - lr: 1.00e-03
Epoch 2/15 - loss: 0.016072 - lr: 1.00e-03
Epoch 3/15 - loss: 0.010293 - lr: 1.00e

,auc_roc,eer,f1
"skip=False, agg=mean",0.631994,0.429897,0.301165
"skip=False, agg=max",0.631785,0.433027,0.307820
"skip=True, agg=mean",0.630803,0.426442,0.322795
"skip=True, agg=max",0.628065,0.436667,0.314476


## 8. Persist baseline

In [10]:
BASELINE_NAME = "skip=False, agg=mean"
model = ablation_models[BASELINE_NAME]
results = ablation_results[BASELINE_NAME]

model.eval()
scores_, labels_ = [], []
with torch.no_grad():
    for batch in test_loader:
        clip = batch["clip"].to(device)
        scores_.append(model.anomaly_score(clip).cpu().numpy())
        labels_.append(batch["clip_label"].numpy())
all_scores, all_labels = np.concatenate(scores_), np.concatenate(labels_)

save_checkpoint(model, "conv_ae_baseline")
save_results("conv_ae_baseline", results, all_scores, all_labels)
print("Baseline persisted as conv_ae_baseline (checkpoint + metrics + scores).")

Baseline persisted as conv_ae_baseline (checkpoint + metrics + scores).


In [12]:
# %cd "{REPO_DIR}"
# !git add notebooks/04_ucsd_model_experiments.ipynb
# !git commit -m "feat: persist ConvAE ablation + baseline checkpoints and results"
# !git push

/content/drive/MyDrive/SurveillanceAnomalyDetection/repo
[main 31f2063] feat: persist ConvAE ablation + baseline checkpoints and results
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite notebooks/04_ucsd_model_experiments.ipynb (97%)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 3.17 KiB | 649.00 KiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Rishabh-G-Shetye/SurveillanceAnomalyDetection.git
   dae8778..31f2063  main -> main
